In [1]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

In [2]:
llm.invoke("how can langsmith help with testing?")

AIMessage(content='Langsmith can help with testing in the following ways:\n\n1. Automated testing: Langsmith can automate the testing process by writing test scripts in various programming languages and executing them to ensure the functionality of the software.\n\n2. Test case generation: Langsmith can generate test cases based on the requirements and specifications of the software, helping to identify potential issues and defects.\n\n3. Test data generation: Langsmith can generate test data to be used in testing, ensuring that the software performs as expected under different scenarios and conditions.\n\n4. Test execution: Langsmith can execute tests and provide detailed reports on the results, making it easier for testers to identify and fix any issues.\n\n5. Integration testing: Langsmith can help with integration testing by simulating the interaction between different components of the software to ensure that they work together seamlessly.\n\nOverall, Langsmith can streamline the 

In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a world class technical documentation writer."),
        ("user", "{input}"),
    ]
)

In [4]:
chain = prompt | llm

In [5]:
chain.invoke({"input": "how can langsmith help with testing?"})

AIMessage(content='Langsmith can assist with testing in several ways:\n\n1. **Automated Testing:** Langsmith can be used to generate test data and test cases automatically, saving time and effort in writing test scripts and scenarios.\n\n2. **Performance Testing:** Langsmith can help simulate large volumes of data to test the performance of systems and applications under different load conditions.\n\n3. **Data Validation:** Langsmith can be used to validate data integrity and consistency across systems by generating test data for various scenarios.\n\n4. **Regression Testing:** Langsmith can help in automating regression testing by generating test data that covers different test cases and scenarios to ensure that new changes do not break existing functionality.\n\n5. **Security Testing:** Langsmith can assist in security testing by generating test data that includes various edge cases and scenarios to identify vulnerabilities in the system.\n\nOverall, Langsmith can streamline the test

In [6]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [7]:
chain = prompt | llm | output_parser

In [8]:
chain.invoke({"input": "how can langsmith help with testing?"})

'Langsmith is a powerful tool that can significantly aid in testing processes. Here are some ways in which Langsmith can help with testing:\n\n1. **Automated Testing**: Langsmith can be used to automate various testing tasks, such as running test cases, generating test data, and comparing expected and actual results. This can help save time and effort in the testing process.\n\n2. **Test Data Generation**: Langsmith can be used to generate synthetic test data to simulate different scenarios and edge cases. This can help in testing the robustness and reliability of the system under various conditions.\n\n3. **Code Analysis**: Langsmith can analyze the codebase for potential issues such as code smells, duplication, complexity, and potential bugs. This can help in identifying areas that require further testing and optimization.\n\n4. **Performance Testing**: Langsmith can be used to analyze the performance of the system by measuring response times, throughput, and resource utilization. Th

In [9]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.smith.langchain.com/user_guide")

docs = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [10]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [11]:
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter()
documents = text_splitter.split_documents(docs)
vector = FAISS.from_documents(documents, embeddings)

In [12]:
from langchain.chains.combine_documents import create_stuff_documents_chain

prompt = ChatPromptTemplate.from_template(
    """
    Answer the follwing question based only on the provieded context:

    <context>
    {context}
    <context>

    Question: {input}
"""
)

document_chain = create_stuff_documents_chain(llm, prompt)

In [13]:
from langchain_core.documents import Document

document_chain.invoke(
    {
        "input": "how can langsmith help with testing?",
        "context": [
            Document(page_content="langsmith can let you visualize test results")
        ],
    }
)

'Langsmith can help with testing by allowing you to visualize test results.'

In [14]:
from langchain.chains import create_retrieval_chain

retriever = vector.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [15]:
response = retrieval_chain.invoke({"input": "how can langsmith help with testing?"})
print(response["answer"])

# LangSmith offers several features that can help with testing:...

LangSmith can help with testing by allowing developers to create datasets, run tests on their LLM applications, upload test cases in bulk, create test cases on the fly, export test cases from application traces, and run custom evaluations to score test results. Additionally, LangSmith provides a comparison view to track and diagnose regressions in test scores across multiple revisions of an application, a playground environment for rapid iteration and experimentation, and features for beta testing, capturing feedback, annotating traces, and adding runs to datasets for refining and improving application performance.


In [16]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder

# First we need a prompt that we can pass into an LLM to generate this search query

prompt = ChatPromptTemplate.from_messages(
    [
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
        (
            "user",
            "Given the above conversation, generate a search query to look up to get information relevant to the conversation",
        ),
    ]
)
retriever_chain = create_history_aware_retriever(llm, retriever, prompt)

In [17]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage(content="Can LangSmith help test my LLM applications?"),
    AIMessage(content="Yes!"),
]
retriever_chain.invoke(
    {
        "chat_history": chat_history,
        "input": "Tell me how",
    }
)

[Document(metadata={'source': 'https://docs.smith.langchain.com/user_guide', 'title': 'LangSmith User Guide | 🦜️🛠️ LangSmith', 'description': 'LangSmith is a platform for LLM application development, monitoring, and testing. In this guide, we’ll highlight the breadth of workflows LangSmith supports and how they fit into each stage of the application development lifecycle. We hope this will inform users how to best utilize this powerful platform or give them something to consider if they’re just starting their journey.', 'language': 'en'}, page_content='Skip to main contentGo to API DocsSearchGo to AppQuick StartUser GuideTracingEvaluationProduction Monitoring & AutomationsPrompt HubProxyPricingSelf-HostingCookbookThis is outdated documentation for 🦜️🛠️ LangSmith, which is no longer actively maintained.For up-to-date documentation, see the latest version.User GuideOn this pageLangSmith User GuideLangSmith is a platform for LLM application development, monitoring, and testing. In this gu

In [18]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the user's questions based on the below context:\n\n{context}",
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("user", "{input}"),
    ]
)
document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever_chain, document_chain)

In [19]:
chat_history = [
    HumanMessage(content="Can LangSmith help test my LLM applications?"),
    AIMessage(content="Yes!"),
]
retrieval_chain.invoke(
    {
        "chat_history": chat_history,
        "input": "Tell me how",
    }
)

{'chat_history': [HumanMessage(content='Can LangSmith help test my LLM applications?'),
  AIMessage(content='Yes!')],
 'input': 'Tell me how',
 'context': [Document(metadata={'source': 'https://docs.smith.langchain.com/user_guide', 'title': 'LangSmith User Guide | 🦜️🛠️ LangSmith', 'description': 'LangSmith is a platform for LLM application development, monitoring, and testing. In this guide, we’ll highlight the breadth of workflows LangSmith supports and how they fit into each stage of the application development lifecycle. We hope this will inform users how to best utilize this powerful platform or give them something to consider if they’re just starting their journey.', 'language': 'en'}, page_content='Skip to main contentGo to API DocsSearchGo to AppQuick StartUser GuideTracingEvaluationProduction Monitoring & AutomationsPrompt HubProxyPricingSelf-HostingCookbookThis is outdated documentation for 🦜️🛠️ LangSmith, which is no longer actively maintained.For up-to-date documentation, se

In [20]:
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "Search for information about LangSmith. For any questions about LangSmith, you must use this tool!",
)

In [21]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults()

In [22]:
tools = [retriever_tool, search]

In [23]:
from langchain_openai import ChatOpenAI
from langchain import hub
from langchain.agents import create_openai_functions_agent
from langchain.agents import AgentExecutor

# Get the prompt to use - you can modify this!
prompt = hub.pull("hwchase17/openai-functions-agent")

# You need to set OPENAI_API_KEY environment variable or pass it as argument `api_key`.
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
agent = create_openai_functions_agent(llm, tools, prompt)
agent_excutor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
)

In [24]:
agent_excutor.invoke({"input": "how can langsmith help with testing?"})



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'how can LangSmith help with testing'}`


Skip to main contentGo to API DocsSearchGo to AppQuick StartUser GuideTracingEvaluationProduction Monitoring & AutomationsPrompt HubProxyPricingSelf-HostingCookbookThis is outdated documentation for 🦜️🛠️ LangSmith, which is no longer actively maintained.For up-to-date documentation, see the latest version.User GuideOn this pageLangSmith User GuideLangSmith is a platform for LLM application development, monitoring, and testing. In this guide, we’ll highlight the breadth of workflows LangSmith supports and how they fit into each stage of the application development lifecycle. We hope this will inform users how to best utilize this powerful platform or give them something to consider if they’re just starting their journey.Prototyping​Prototyping LLM applications often involves quick experimentation between prompts, model types, retrieval strategy and other parame

{'input': 'how can langsmith help with testing?',
 'output': "LangSmith can help with testing in various ways throughout the application development lifecycle:\n\n1. **Prototyping**: LangSmith supports prototyping LLM applications by allowing quick experimentation between prompts, model types, retrieval strategies, and other parameters.\n\n2. **Debugging**: LangSmith provides tracing capabilities that enable developers to debug issues in LLM applications by offering clear visibility and debugging information at each step of the sequence.\n\n3. **Initial Test Set**: Developers can create datasets of inputs and reference outputs to run tests on LLM applications. LangSmith makes it easy to run custom evaluations to score test results.\n\n4. **Comparison View**: LangSmith offers a comparison view for test runs, allowing developers to track and diagnose regressions in test scores across multiple revisions of the application.\n\n5. **Playground**: LangSmith provides a playground environment 

In [25]:
agent_excutor.invoke({"input": "what is the weather in SF?"})



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'weather in San Francisco'}`


[{'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'San Francisco', 'region': 'California', 'country': 'United States of America', 'lat': 37.78, 'lon': -122.42, 'tz_id': 'America/Los_Angeles', 'localtime_epoch': 1720523650, 'localtime': '2024-07-09 4:14'}, 'current': {'last_updated_epoch': 1720522800, 'last_updated': '2024-07-09 04:00', 'temp_c': 16.1, 'temp_f': 61.0, 'is_day': 0, 'condition': {'text': 'Overcast', 'icon': '//cdn.weatherapi.com/weather/64x64/night/122.png', 'code': 1009}, 'wind_mph': 6.9, 'wind_kph': 11.2, 'wind_degree': 240, 'wind_dir': 'WSW', 'pressure_mb': 1011.0, 'pressure_in': 29.86, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 90, 'cloud': 100, 'feelslike_c': 16.1, 'feelslike_f': 61.0, 'windchill_c': 13.0, 'windchill_f': 55.4, 'heatindex_c': 14.1, 'heatindex_f': 57.4, 'dewpoint_c': 12.3, 'dewpoint_f': 54.1, 'vis_k

{'input': 'what is the weather in SF?',
 'output': 'The current weather in San Francisco is overcast with a temperature of 61.0°F (16.1°C). The wind is coming from the WSW direction at 11.2 km/h. The humidity is at 90%, and the visibility is 9.0 miles.'}

In [26]:
chat_history = [
    HumanMessage(content="Can LangSmith help test my LLM applications?"),
    AIMessage(content="Yes!"),
]
agent_excutor.invoke(
    {"chat_history": chat_history, "input": "Tell me how"},
)



> Entering new AgentExecutor chain...

Invoking: `langsmith_search` with `{'query': 'LangSmith LLM application testing services'}`


LangSmith User Guide | 🦜️🛠️ LangSmith

Skip to main contentGo to API DocsSearchGo to AppQuick StartUser GuideTracingEvaluationProduction Monitoring & AutomationsPrompt HubProxyPricingSelf-HostingCookbookThis is outdated documentation for 🦜️🛠️ LangSmith, which is no longer actively maintained.For up-to-date documentation, see the latest version.User GuideOn this pageLangSmith User GuideLangSmith is a platform for LLM application development, monitoring, and testing. In this guide, we’ll highlight the breadth of workflows LangSmith supports and how they fit into each stage of the application development lifecycle. We hope this will inform users how to best utilize this powerful platform or give them something to consider if they’re just starting their journey.Prototyping​Prototyping LLM applications often involves quick experimentation between prompts, mo

{'chat_history': [HumanMessage(content='Can LangSmith help test my LLM applications?'),
  AIMessage(content='Yes!')],
 'input': 'Tell me how',
 'output': 'LangSmith offers a platform for LLM application development, monitoring, and testing. Here are some ways LangSmith can help test your LLM applications:\n\n1. Prototyping: LangSmith supports prototyping LLM applications by allowing quick experimentation between prompts, model types, and retrieval strategies.\n\n2. Debugging: LangSmith provides tracing capabilities to help developers debug issues in their LLM applications, offering clear visibility and debugging information at each step of an LLM sequence.\n\n3. Initial Test Set: Developers can create datasets of inputs and reference outputs to run tests on their LLM applications, enabling them to evaluate the performance of their applications.\n\n4. Comparison View: LangSmith offers a comparison view for test runs, allowing developers to track and diagnose regressions in test scores a